In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n\n{device}")

/kaggle/input/ai-powered-resume-screening-dataset-2025/AI_Resume_Screening.csv
/kaggle/input/audio-tracks/5 . 14.22_.m4a
/kaggle/input/for-llama/2   .docx
/kaggle/input/for-llama/1   .docx
/kaggle/input/for-llama/1  .rtf
/kaggle/input/for-llama/.docx
/kaggle/input/for-llama/2  .rtf
/kaggle/input/resume-dataset/UpdatedResumeDataSet.csv
/kaggle/input/fine-tuning/kaggle/working/qlora-mistral7b-hr-v4/lora_adapter660/adapter_model.safetensors
/kaggle/input/fine-tuning/kaggle/working/qlora-mistral7b-hr-v4/lora_adapter660/adapter_config.json
/kaggle/input/fine-tuning/kaggle/working/qlora-mistral7b-hr-v4/lora_adapter660/README.md
/kaggle/input/fine-tuning/kaggle/working/qlora-mistral7b-hr-v4/lora_adapter660/tokenizer.json
/kaggle/input/fine-tuning/kaggle/working/qlora-mistral7b-hr-v4/lora_adapter660/tokenizer_config.json
/kaggle/input/fine-tuning/kaggle/working/qlora-mistral7b-hr-v4/lora_adapter660/special_tokens_map.json
/kaggle/input/fine-tuning/kaggle/working/qlora-mistral7b-hr-v4/lora_adap

In [4]:
!nvidia-smi

Fri Sep  5 18:08:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Install latest bitsandbytes & transformers, accelerate from source
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
# Other requirements for the demo
!pip install gradio
!pip install sentencepiece

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
!pip uninstall -y transformers peft accelerate
!pip install -q transformers==4.44.2 peft==0.11.1 accelerate==0.33.0 sentence-transformers striprtf python-docx mammoth

Found existing installation: transformers 4.57.0.dev0
Uninstalling transformers-4.57.0.dev0:
  Successfully uninstalled transformers-4.57.0.dev0
Found existing installation: peft 0.17.2.dev0
Uninstalling peft-0.17.2.dev0:
  Successfully uninstalled peft-0.17.2.dev0
Found existing installation: accelerate 1.11.0.dev0
Uninstalling accelerate-1.11.0.dev0:
  Successfully uninstalled accelerate-1.11.0.dev0


In [7]:
!pip install faster-whisper --quiet

In [27]:
import json
from pathlib import Path
from typing import Dict, Any

from faster_whisper import WhisperModel

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForAudioClassification,
    pipeline,
    Wav2Vec2FeatureExtractor,
    AutoProcessor,
    Wav2Vec2ForSequenceClassification
)
import librosa
print("Готово")

Готово


In [34]:
# ASR (faster-whisper small)
asr_model = WhisperModel("small", device=device, compute_type="float16")
!nvidia-smi


Fri Sep  5 18:25:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |    2777MiB /  15360MiB |     22%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
# Text emotion
text_emotion_pipeline = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=0 if device=="cuda" else -1
)
!nvidia-smi

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Fri Sep  5 18:25:34 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |    2893MiB /  15360MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [38]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

voice_model_name = "superb/wav2vec2-base-superb-er"

#feature extractor
voice_processor = AutoFeatureExtractor.from_pretrained(voice_model_name)


voice_model = AutoModelForAudioClassification.from_pretrained(voice_model_name).to(device)

# Метки эмоций
voice_labels = voice_model.config.id2label

Some weights of the model checkpoint at superb/wav2vec2-base-superb-er were not used when initializing Wav2Vec2ForSequenceClassification: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at superb/wav2vec2-base-superb-er and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_em

In [40]:
!nvidia-smi

Fri Sep  5 18:31:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             26W /   70W |    2967MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [32]:
class SpeechToTextEmotion:
    def __init__(self, asr_model, text_emotion_pipeline, voice_model, voice_processor, voice_labels, device="cuda"):
        self.device = device
        self.asr_model = asr_model
        self.text_emotion = text_emotion_pipeline
        self.voice_model = voice_model
        self.voice_processor = voice_processor
        self.voice_labels = voice_labels

    def transcribe(self, audio_path: str) -> str:
        # faster-whisper возвращает генератор сегментов + инфо
        segments, _ = self.asr_model.transcribe(audio_path)
        text = " ".join([seg.text for seg in segments])
        return text.strip()

    def analyze_text_emotion(self, text: str) -> Dict[str, Any]:
        preds = self.text_emotion(text, top_k=1)
        # pipeline возвращает список списков
        pred = preds[0][0] if isinstance(preds[0], list) else preds[0]
        return {"label": pred["label"], "conf": float(pred["score"])}

    def analyze_voice_emotion(self, audio_path: str) -> Dict[str, Any]:
        speech, sr = librosa.load(audio_path, sr=16000)

        inputs = self.voice_processor(
            speech,
            sampling_rate=sr,
            return_tensors="pt"
        )

        with torch.no_grad():
            logits = self.voice_model(inputs["input_values"].to(self.device)).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

        label_id = int(probs.argmax())
        return {
            "label": self.voice_labels[label_id],
            "conf": float(probs[label_id])
        }

    def fuse_emotions(self, voice: Dict[str, Any], text: Dict[str, Any]) -> str:
        if text["conf"] > 0.7:
            return text["label"]
        return voice["label"]

    def process(self, audio_path: str) -> Dict[str, Any]:
        text = self.transcribe(audio_path)
        emo_text = self.analyze_text_emotion(text)
        emo_voice = self.analyze_voice_emotion(audio_path)
        final = self.fuse_emotions(emo_voice, emo_text)

        return {
            "Text": text,
            "Emotion": {
                "voice": emo_voice,
                "text": emo_text,
                "final": final
            }
        }


print("Готово")

Готово


In [39]:
engine = SpeechToTextEmotion(
    asr_model=asr_model,
    text_emotion_pipeline=text_emotion_pipeline,
    voice_model=voice_model,
    voice_processor=voice_processor,
    voice_labels=voice_labels,
    device=device
)

audio_path = "/kaggle/input/audio-tracks/5 . 14.22_.m4a"
result = engine.process(audio_path)
print(result)

/tmp/ipykernel_567/577640749.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  speech, sr = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


{'Text': 'Работать в команде, ну, наиболее комфортно работаю, когда каждый команды понимает, что делает и почему. Кроме того, мне нравится открыто обсуждение идей, критика и поддержка коллег. Также играет важную роль в успешном сотруднице. В целом, это то, что у меня всегда хорошо получается, и я рад буду присоединиться к вашей команде.', 'Emotion': {'voice': {'label': 'hap', 'conf': 0.9463099241256714}, 'text': {'label': 'neutral', 'conf': 0.5453219413757324}, 'final': 'hap'}}
